# VPC & Networking

## VPC Basics

A **VPC** is a logically isolated network in AWS. You define a CIDR block (e.g., 10.0.0.0/16) and create subnets within it.

**Subnets** are subdivisions of a VPC within a single availability zone. A public subnet has a route to the internet; a private subnet does not.

**Route tables** define how traffic is routed. Each subnet is associated with a route table that specifies where traffic destined for different CIDR blocks should go.

## Internet Gateway and NAT

An **Internet Gateway** allows resources in a public subnet to communicate with the internet. Attach it to your VPC and add a route to it in the route table.

A **NAT Gateway** allows resources in a private subnet to initiate outbound connections to the internet without receiving inbound connections. It's placed in a public subnet and uses an Elastic IP.

## Security Groups vs Network ACLs

**Security Groups** are stateful firewalls at the instance level. If you allow inbound traffic, outbound response is automatically allowed.

**Network ACLs (NACLs)** are stateless firewalls at the subnet level. You must explicitly allow both inbound and outbound traffic. NACLs are evaluated before security groups.

## Hands-On: Create VPC with Public and Private Subnets

Create a VPC:

```bash
aws ec2 create-vpc --cidr-block 10.0.0.0/16
```

Create a public subnet:

```bash
aws ec2 create-subnet --vpc-id vpc-xxxxx --cidr-block 10.0.1.0/24 \
  --availability-zone us-east-1a
```

Create a private subnet:

```bash
aws ec2 create-subnet --vpc-id vpc-xxxxx --cidr-block 10.0.2.0/24 \
  --availability-zone us-east-1a
```

Create an Internet Gateway:

```bash
aws ec2 create-internet-gateway
aws ec2 attach-internet-gateway --internet-gateway-id igw-xxxxx \
  --vpc-id vpc-xxxxx
```

Create a route table for public subnet:

```bash
aws ec2 create-route-table --vpc-id vpc-xxxxx
aws ec2 create-route --route-table-id rtb-xxxxx \
  --destination-cidr-block 0.0.0.0/0 \
  --gateway-id igw-xxxxx
```

Associate route table with public subnet:

```bash
aws ec2 associate-route-table --subnet-id subnet-xxxxx \
  --route-table-id rtb-xxxxx
```

## Python Boto3 Example

In [ ]:
import boto3

ec2 = boto3.client('ec2')

# Create VPC
vpc = ec2.create_vpc(CidrBlock='10.0.0.0/16')
vpc_id = vpc['Vpc']['VpcId']

# Create subnet
subnet = ec2.create_subnet(VpcId=vpc_id, CidrBlock='10.0.1.0/24')
subnet_id = subnet['Subnet']['SubnetId']

# Create Internet Gateway
igw = ec2.create_internet_gateway()
igw_id = igw['InternetGateway']['InternetGatewayId']

# Attach IGW to VPC
ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)

# Create route table
rt = ec2.create_route_table(VpcId=vpc_id)
rt_id = rt['RouteTable']['RouteTableId']

# Add route to IGW
ec2.create_route(
    RouteTableId=rt_id,
    DestinationCidrBlock='0.0.0.0/0',
    GatewayId=igw_id
)

# Associate route table with subnet
ec2.associate_route_table(SubnetId=subnet_id, RouteTableId=rt_id)

## Terraform Example

```hcl
resource "aws_vpc" "main" {
  cidr_block = "10.0.0.0/16"
}

resource "aws_subnet" "public" {
  vpc_id            = aws_vpc.main.id
  cidr_block        = "10.0.1.0/24"
  availability_zone = "us-east-1a"
}

resource "aws_internet_gateway" "main" {
  vpc_id = aws_vpc.main.id
}

resource "aws_route_table" "public" {
  vpc_id = aws_vpc.main.id

  route {
    cidr_block      = "0.0.0.0/0"
    gateway_id      = aws_internet_gateway.main.id
  }
}

resource "aws_route_table_association" "public" {
  subnet_id      = aws_subnet.public.id
  route_table_id = aws_route_table.public.id
}
```

## VPC Flow Logs

Enable VPC Flow Logs to monitor network traffic:

```bash
aws ec2 create-flow-logs --resource-type VPC \
  --resource-ids vpc-xxxxx --traffic-type ALL \
  --log-destination-type cloud-watch-logs \
  --log-group-name /aws/vpc/flowlogs
```

## Quiz 1

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is a VPC?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="0">
      <span>A logically isolated network in AWS</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="1">
      <span>A virtual machine</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="2">
      <span>A storage service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="3">
      <span>A database</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the purpose of a NAT Gateway?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="0">
      <span>To route traffic between VPCs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="1">
      <span>To allow private subnet resources to access the internet</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="2">
      <span>To encrypt network traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="3">
      <span>To monitor network activity</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is the key difference between Security Groups and NACLs?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="0">
      <span>Security Groups are stateful; NACLs are stateless</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="1">
      <span>NACLs are stateful; Security Groups are stateless</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="2">
      <span>They are identical</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="3">
      <span>Security Groups work at region level; NACLs at VPC level</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What does an Internet Gateway do?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="0">
      <span>Encrypts traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="1">
      <span>Stores data</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="2">
      <span>Allows resources in public subnets to communicate with the internet</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="3">
      <span>Manages user access</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is a route table?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="0">
      <span>A list of users with access to resources</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="1">
      <span>A set of rules defining how traffic is routed</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="2">
      <span>A database table</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="3">
      <span>A backup configuration</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>